# Purpose of This Notebook

This notebook prepares curated, analysis-ready datasets for visualization and reporting in Tableau.

After completing exploratory data analysis, statistical testing, and predictive modeling, this notebook serves as the final data layer that translates our findings into clear, stakeholder-facing visuals.

## Why This Matters
- Our Critical Research Question asks: Do neighborhoods with higher shelter-related food insecurity prioritization have sufficient emergency food assistance supply?
- To answer this effectively for policymakers and stakeholders, we need structured visual outputs that show:
    - 📍 Neighborhood-level need (prioritization scores, food insecurity rates)
    - 🥫 Supply of EFAP sites (total sites, pantry vs kitchen, weekend access)
    - 🏠 Shelter demand context (families with children in shelters)
    - ⚖️ Mismatch between high need and low supply
- This notebook generates the SQL queries used to extract and join these components from our star schema database


## What This Notebook Does
**Connects to the food_access.db database and joins fact and dimension tables:**
- fact_food_site_coverage
- fact_neighborhood_prioritization
- fact_agg_shelter_cdta_year
- dim_map
- dim_date

**Produces Tableau-ready datasets for:**
- High-need vs supply comparisons
- Underserved neighborhood rankings
- Shelter demand vs EFAP supply analysis
- Borough-level summaries


## workflow
- EDA → Statistical Analysis → Modeling → Dashboard Data Layer (This Notebook) → Tableau Visualization → Final Presentation

In [2]:
import sqlite3
import pandas as pd

# Path to your database
db_path = "/Users/ayemaq/Desktop/CID_Food_Access/data/food_access.db"  

conn = sqlite3.connect(db_path)

### Visual 1) high need neighborhoods vs supply
- Chart idea: Bar chart or scatter: prioritization (need) vs total sites (supply)  
- Tableau use:
    - x-axis: weighted_score or food_insecure_percentage
    - y-axis: total_sites
    - color: boro_name or supply_gap
    - filter: year

In [3]:
query_1 = """
SELECT
  m.boro_name,
  f.nta_id,
  m.nta_name,
  d.year,
  p.weighted_score,
  p.food_insecure_percentage,
  p.supply_gap,
  f.total_sites,
  f.pantry_site_count,
  f.kitchen_site_count,
  f.weekend_site_count
FROM fact_food_site_coverage f
JOIN dim_date d ON d.date_id = f.date_id
JOIN dim_map m ON m.nta_id = f.nta_id
JOIN fact_neighborhood_prioritization p ON p.nta_id = f.nta_id
WHERE d.year IN (2023, 2024);
"""
df_need_vs_supply = pd.read_sql_query(query_1, conn)

df_need_vs_supply.head()

,boro_name,nta_id,nta_name,year,weighted_score,food_insecure_percentage,supply_gap,total_sites,pantry_site_count,kitchen_site_count,weekend_site_count
0,Brooklyn,BK0101,Greenpoint,2024,5.6492,12.63,6.563384e+05,1,1,0,0
1,Brooklyn,BK0102,Williamsburg,2024,6.1449,16.60,8.220354e+05,1,1,0,0
2,Brooklyn,BK0103,South Williamsburg,2024,6.4041,14.54,7.186889e+05,1,1,0,0
3,Brooklyn,BK0104,East Williamsburg,2024,8.2210,35.99,2.776626e+06,1,1,0,0
4,Brooklyn,BK0202,Downtown Brooklyn-DUMBO-Boerum Hill,2024,4.5585,13.93,-2.906213e+05,3,2,1,1


### Visual 2: The mismatch list (top underserved NTAs)
- Chart idea: ranked table or bar: “highest need, lowest supply”

In [4]:
query_2 = """
SELECT
  d.year,
  m.boro_name,
  f.nta_id,
  m.nta_name,
  p.weighted_score,
  p.food_insecure_percentage,
  p.supply_gap,
  f.total_sites,
  f.kitchen_site_count,
  f.weekend_site_count
FROM fact_food_site_coverage f
JOIN dim_date d ON d.date_id = f.date_id
JOIN dim_map m ON m.nta_id = f.nta_id
JOIN fact_neighborhood_prioritization p ON p.nta_id = f.nta_id
WHERE d.year IN (2023, 2024)
ORDER BY p.weighted_score DESC, f.total_sites ASC
LIMIT 25;
"""

df_mismatch = pd.read_sql_query(query_2, conn)

df_mismatch.head()

,year,boro_name,nta_id,nta_name,weighted_score,food_insecure_percentage,supply_gap,total_sites,kitchen_site_count,weekend_site_count
0,2024,Brooklyn,BK0104,East Williamsburg,8.2210,35.99,2.776626e+06,1,0,0
1,2024,Bronx,BX0501,University Heights (South)-Morris Heights,8.0704,29.44,1.669389e+06,3,0,0
2,2024,Bronx,BX0901,Soundview-Bruckner-Bronx River,7.6866,22.63,1.625976e+06,4,1,3
3,2024,Manhattan,MN1202,Washington Heights (North),7.3895,24.29,1.463457e+06,2,1,0
4,2024,Brooklyn,BK1503,Sheepshead Bay-Manhattan Beach-Gerritsen Beach,7.2775,21.11,1.907056e+06,1,1,0


### Visual 3: Shelter demand context added (CDTA families)
- Chart idea: Scatterplot: shelter families vs total sites (supply)
- Tableau use:
    - x-axis: family_with_children_count
    - y-axis: total_sites
    - color: weighted_score
    - filter: borough/yeair


In [5]:
query_3 = """ 
WITH shelter AS (
  SELECT
    s.cdta_id, d.year, s.family_with_children_count
  FROM fact_agg_shelter_cdta_year s
  JOIN dim_date d ON d.date_id = s.date_id
  WHERE d.year IN (2023, 2024)
)
SELECT
  sh.year,
  m.cdta_id,
  m.cdta_name,
  m.boro_name,
  m.nta_id,
  m.nta_name,
  sh.family_with_children_count,
  f.total_sites,
  p.weighted_score
FROM shelter sh
JOIN dim_map m ON m.cdta_id = sh.cdta_id
JOIN fact_food_site_coverage f ON f.nta_id = m.nta_id
JOIN dim_date d2 ON d2.date_id = f.date_id AND d2.year = sh.year
JOIN fact_neighborhood_prioritization p ON p.nta_id = m.nta_id;

"""

df_cdta_demand_supply = pd.read_sql_query(query_3, conn)

df_cdta_demand_supply.head()

,year,cdta_id,cdta_name,boro_name,nta_id,nta_name,family_with_children_count,total_sites,weighted_score
0,2024,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),Brooklyn,BK0101,Greenpoint,0,1,5.6492
1,2024,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),Brooklyn,BK0102,Williamsburg,0,1,6.1449
2,2024,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),Brooklyn,BK0103,South Williamsburg,0,1,6.4041
3,2024,BK01,BK01 Williamsburg-Greenpoint (CD 1 Equivalent),Brooklyn,BK0104,East Williamsburg,0,1,8.2210
4,2024,BK02,BK02 Downtown Brooklyn-Fort Greene (CD 2 Appro...,Brooklyn,BK0202,Downtown Brooklyn-DUMBO-Boerum Hill,853,3,4.5585


### Visual 4: Access type breakdown (kitchen vs pantry vs weekend)
- Chart idea: Stacked bar by borough or year
- Great for showing operational capacity differences by borough!!

In [6]:
query_4 = """ 
SELECT
  d.year,
  m.boro_name,
  SUM(f.total_sites) AS total_sites,
  SUM(f.pantry_site_count) AS pantry_sites,
  SUM(f.kitchen_site_count) AS kitchen_sites,
  SUM(f.weekend_site_count) AS weekend_sites
FROM fact_food_site_coverage f
JOIN dim_date d ON d.date_id = f.date_id
JOIN dim_map m ON m.nta_id = f.nta_id
WHERE d.year IN (2023, 2024)
GROUP BY d.year, m.boro_name
ORDER BY d.year, m.boro_name;
"""

df_access_type = pd.read_sql_query(query_4, conn)
df_access_type.head()

,year,boro_name,total_sites,pantry_sites,kitchen_sites,weekend_sites
0,2024,Bronx,108,94,15,37
1,2024,Brooklyn,195,171,39,64
2,2024,Manhattan,103,73,32,29
3,2024,Queens,126,117,17,44
4,2024,Staten Island,21,17,6,6


### Visual 5: “No Access” neighborhoods -- NOT NEEDED
- Chart idea: Map or table: NTAs with 0 sites but discuss this with team

In [7]:
query_5 = """ 
SELECT
  d.year,
  m.boro_name,
  f.nta_id,
  m.nta_name,
  p.weighted_score,
  p.supply_gap,
  f.total_sites
FROM fact_food_site_coverage f
JOIN dim_date d ON d.date_id = f.date_id
JOIN dim_map m ON m.nta_id = f.nta_id
JOIN fact_neighborhood_prioritization p ON p.nta_id = f.nta_id
WHERE d.year IN (2023, 2024)
  AND f.total_sites = 0
ORDER BY d.year, p.weighted_score DESC;

"""
df_no_access = pd.read_sql_query(query_5, conn)
df_no_access.head()

,year,boro_name,nta_id,nta_name,weighted_score,supply_gap,total_sites
